In [2]:
%pip install shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 3.2 MB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 3.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [shap]4/5 [shap]]te]
Note: you may need to restart the kernel to use updated packages.


In [3]:
# ============================================
# SHAP MODEL EXPLAINABILITY
# ============================================

import pandas as pd
import numpy as np
import shap

print("SHAP version:", shap.__version__)

/opt/anaconda3/envs/jupyter-ds/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP version: 0.52.0


In [4]:
# ============================================
# LOAD AND PREPARE DATA
# ============================================

import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Remove invalid rows
df.dropna(subset=["TotalCharges"], inplace=True)

# Remove duplicate rows
df.drop_duplicates(inplace=True)

# Remove customer ID
df.drop(columns=["customerID"], inplace=True)

# Convert target to 0/1
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

print("Dataset prepared successfully.")
print("Shape:", df.shape)

Dataset prepared successfully.
Shape: (7032, 20)


In [5]:
# ============================================
# SEPARATE FEATURES AND TARGET
# ============================================

X = df.drop("Churn", axis=1)
y = df["Churn"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (7032, 19)
Target shape: (7032,)


In [6]:
# ============================================
# IDENTIFY FEATURE TYPES
# ============================================

numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']

Categorical Features:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [7]:
# ============================================
# CREATE PREPROCESSING PIPELINE
# ============================================

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    drop="first"
)

numerical_transformer = StandardScaler()

preprocessor_scaled = ColumnTransformer(
    transformers=[
        ("categorical", categorical_transformer, categorical_features),
        ("numerical", numerical_transformer, numerical_features)
    ]
)

print("Preprocessing pipeline created successfully.")

Preprocessing pipeline created successfully.


In [8]:
# ============================================
# TRAIN / TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

Training features: (5625, 19)
Testing features: (1407, 19)


In [9]:
# ============================================
# LOGISTIC REGRESSION MODEL
# ============================================

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(
            solver="liblinear",
            max_iter=1000,
            random_state=42
        ))
    ]
)

logistic_pipeline.fit(X_train, y_train)

print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


In [10]:
# ============================================
# VERIFY MODEL PERFORMANCE
# ============================================

from sklearn.metrics import roc_auc_score

y_test_proba = logistic_pipeline.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_test_proba)

print(f"Test ROC-AUC: {roc_auc:.4f}")

Test ROC-AUC: 0.8362


In [11]:
# ============================================
# TRANSFORM TEST DATA FOR SHAP
# ============================================

X_test_transformed = logistic_pipeline.named_steps[
    "preprocessor"
].transform(X_test)

print("Original test shape:", X_test.shape)
print("Transformed test shape:", X_test_transformed.shape)

Original test shape: (1407, 19)
Transformed test shape: (1407, 30)


In [12]:
# ============================================
# GET TRANSFORMED FEATURE NAMES
# ============================================

feature_names = logistic_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

print("Number of transformed features:", len(feature_names))
print("\nFirst 20 feature names:")
print(feature_names[:20])

Number of transformed features: 30

First 20 feature names:
['categorical__gender_Male' 'categorical__Partner_Yes'
 'categorical__Dependents_Yes' 'categorical__PhoneService_Yes'
 'categorical__MultipleLines_No phone service'
 'categorical__MultipleLines_Yes'
 'categorical__InternetService_Fiber optic'
 'categorical__InternetService_No'
 'categorical__OnlineSecurity_No internet service'
 'categorical__OnlineSecurity_Yes'
 'categorical__OnlineBackup_No internet service'
 'categorical__OnlineBackup_Yes'
 'categorical__DeviceProtection_No internet service'
 'categorical__DeviceProtection_Yes'
 'categorical__TechSupport_No internet service'
 'categorical__TechSupport_Yes'
 'categorical__StreamingTV_No internet service'
 'categorical__StreamingTV_Yes'
 'categorical__StreamingMovies_No internet service'
 'categorical__StreamingMovies_Yes']


In [13]:
# ============================================
# CREATE SHAP EXPLAINER
# ============================================

model = logistic_pipeline.named_steps["model"]

explainer = shap.LinearExplainer(
    model,
    X_test_transformed,
    feature_names=feature_names
)

print("SHAP LinearExplainer created successfully.")

Background dataset has 1407 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=1407 when initializing the masker.


SHAP LinearExplainer created successfully.


In [14]:
# ============================================
# CALCULATE SHAP VALUES
# ============================================

shap_values = explainer.shap_values(
    X_test_transformed
)

print("SHAP values calculated successfully.")
print("SHAP values shape:", shap_values.shape)

SHAP values calculated successfully.
SHAP values shape: (1407, 30)


In [15]:
# ============================================
# GLOBAL SHAP FEATURE IMPORTANCE
# ============================================

shap_importance = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Absolute_SHAP": np.abs(shap_values).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
)

display(shap_importance.head(15))

,Feature,Mean_Absolute_SHAP
27,numerical__tenure,1.204946
29,numerical__TotalCharges,0.510952
21,categorical__Contract_Two year,0.434225
6,categorical__InternetService_Fiber optic,0.431095
20,categorical__Contract_One year,0.287640
24,categorical__PaymentMethod_Electronic check,0.164301
9,categorical__OnlineSecurity_Yes,0.161912
3,categorical__PhoneService_Yes,0.152895
5,categorical__MultipleLines_Yes,0.152478
15,categorical__TechSupport_Yes,0.143154


In [19]:
# ============================================
# SHAP DIRECTIONAL ANALYSIS
# ============================================

shap_direction = pd.DataFrame({
    "Feature": feature_names,
    "Mean_Absolute_SHAP": np.abs(shap_values).mean(axis=0),
    "Mean_SHAP": shap_values.mean(axis=0),
    "Positive_Impact_%": (shap_values > 0).mean(axis=0) * 100
})

shap_direction = shap_direction.sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
)

display(
    shap_direction.head(15).round(4)
)

,Feature,Mean_Absolute_SHAP,Mean_SHAP,Positive_Impact_%
27,numerical__tenure,1.2049,-0.1151,51.1016
29,numerical__TotalCharges,0.5110,0.0658,39.5878
21,categorical__Contract_Two year,0.4342,-0.0989,76.7591
6,categorical__InternetService_Fiber optic,0.4311,-0.0038,43.5679
20,categorical__Contract_One year,0.2876,0.0706,79.3888
24,categorical__PaymentMethod_Electronic check,0.1643,0.0021,32.5515
9,categorical__OnlineSecurity_Yes,0.1619,-0.0232,71.4996
3,categorical__PhoneService_Yes,0.1529,-0.0259,9.5949
5,categorical__MultipleLines_Yes,0.1525,0.0011,41.3646
15,categorical__TechSupport_Yes,0.1432,-0.0158,72.8500
